# Validation-Only Model Selection

This notebook documents the final model-selection protocol for the persistence-adjusted delta task. It is display-only: all modeling, scoring, artifact writing, and figure generation live in `3_models/scripts/persistence_adjusted_modeling.py`.

## Selection Protocol

Candidate delta models are compared on the validation block after converting predicted deltas back to the original level scale. The selected specification is then fixed before any held-out test interpretation.

The test block is used only for the final generalization check of the validation-selected specifications.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "3_models" / "scripts" / "persistence_adjusted_modeling.py").exists():
            return candidate
    raise RuntimeError("Could not locate project root")


ROOT = _find_project_root(Path(os.getcwd()).resolve())
SCRIPT_DIR = ROOT / "3_models" / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from persistence_adjusted_modeling import (  # noqa: E402
    PERSISTENCE_SELECTION_FIGURE_OUTPUT,
    PERSISTENCE_VALIDATION_LEVEL_CORRECTION_OUTPUT,
    PERSISTENCE_VALIDATION_LEVEL_CORRECTION_SUMMARY_OUTPUT,
    PERSISTENCE_VALIDATION_SELECTED_TEST_LEVEL_CORRECTION_OUTPUT,
    PERSISTENCE_VALIDATION_SELECTED_TEST_LEVEL_CORRECTION_SUMMARY_OUTPUT,
    PERSISTENCE_VALIDATION_SELECTION_OUTPUT,
    load_validation_selection_notebook_tables,
)


tables = load_validation_selection_notebook_tables()


## Artifact Contract

The notebook reads precomputed artifacts. Missing artifacts mean the script pipeline should be regenerated before interpreting model-selection evidence.

Required files: `persistence_adjusted_validation_level_correction_predictions.csv`, `persistence_adjusted_validation_level_correction_summary.csv`, `persistence_adjusted_validation_model_selection.csv`, `persistence_adjusted_validation_selected_test_level_correction_predictions.csv`, and `persistence_adjusted_validation_selected_test_level_correction_summary.csv`.

In [ ]:
artifact_table = pd.DataFrame(
    [
        ("validation_level_predictions", PERSISTENCE_VALIDATION_LEVEL_CORRECTION_OUTPUT),
        ("validation_level_summary", PERSISTENCE_VALIDATION_LEVEL_CORRECTION_SUMMARY_OUTPUT),
        ("validation_model_selection", PERSISTENCE_VALIDATION_SELECTION_OUTPUT),
        ("validation_selected_test_predictions", PERSISTENCE_VALIDATION_SELECTED_TEST_LEVEL_CORRECTION_OUTPUT),
        ("validation_selected_test_summary", PERSISTENCE_VALIDATION_SELECTED_TEST_LEVEL_CORRECTION_SUMMARY_OUTPUT),
        ("validation_selection_figure", PERSISTENCE_SELECTION_FIGURE_OUTPUT),
    ],
    columns=["artifact", "path"],
)
artifact_table["exists"] = artifact_table["path"].map(lambda path: Path(path).exists())
artifact_table["size_bytes"] = artifact_table["path"].map(
    lambda path: Path(path).stat().st_size if Path(path).exists() else 0
)
display(artifact_table)


## Mathematical Target Inversion

For country $i$, the validation anchor is the latest observed training target $y_{i,t_0}$. For a consecutive validation block $t_1, \ldots, t_k$, the model predicts $\widehat{\Delta y}_{i,t_j}$ and the level forecast is reconstructed as:

$$
\widehat{y}^{\mathrm{corr}}_{i,t_k} = y_{i,t_0} + \sum_{j=1}^{k} \widehat{\Delta y}_{i,t_j}.
$$

This is a cumulative sum of predicted deltas, not a cumulative sum of observed validation or test labels.

## Validation Selection Visualization

The left panel shows validation-only selected specifications against the history-only baseline. The right panel shows the after-the-fact test gap for those already selected specifications.

In [ ]:
if PERSISTENCE_SELECTION_FIGURE_OUTPUT.exists():
    display(Image(filename=str(PERSISTENCE_SELECTION_FIGURE_OUTPUT)))
else:
    display(Markdown("Missing validation selection figure."))


## Validation Candidate Ranking

This table ranks candidate specifications by validation corrected-level MAE. It is the only table used for model selection.

In [ ]:
validation_candidates = tables["validation_candidates"]
candidate_columns = [
    "panel_id",
    "lag_suffix",
    "family",
    "best_model",
    "n_forecast",
    "history_only_mae",
    "corrected_level_mae",
    "delta_corrected_mae_minus_history",
    "beats_history_only",
]
display(validation_candidates.loc[:, candidate_columns].head(20).round(4))


## Validation-Selected Final Specs

One specification is selected per target-panel scope using `validation_corrected_level_mae`. These rows are fixed before test interpretation.

In [ ]:
validation_selection = tables["validation_selection"]
selection_columns = [
    "target_variant",
    "panel_id",
    "lag_suffix",
    "family",
    "best_model",
    "history_only_mae",
    "corrected_level_mae",
    "delta_corrected_mae_minus_history",
    "selection_metric",
]
display(validation_selection.loc[:, selection_columns].round(4))


## Validation/Test Generalization Gap

This table evaluates only the already validation-selected specifications on the held-out test block. A large gap indicates temporal instability or a harder late-period test block; it is not used to choose the model.

In [ ]:
validation_test_gap = tables["validation_test_gap"]
gap_columns = [
    "target_variant",
    "panel_id",
    "lag_suffix",
    "family",
    "best_model",
    "corrected_level_mae_validation",
    "corrected_level_mae_test",
    "test_minus_validation_corrected_level_mae",
    "history_only_mae_validation",
    "history_only_mae_test",
    "delta_corrected_mae_minus_history_validation",
    "delta_corrected_mae_minus_history_test",
]
display(validation_test_gap.loc[:, gap_columns].round(4))


## Reviewer Interpretation

Validation corrected-level MAE selects RandomForest delta correction for the main and submodel B panels. The held-out test audit then suggests these selected specifications still improve over the history-only baseline, but with much larger absolute test MAE. The larger test MAE should be reported as a generalization limitation, while the comparison against the history-only baseline remains the practical question: whether predicted deltas add useful temporal signal beyond persistence.